In [1]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

from dotenv import load_dotenv
load_dotenv()
from langchain_openai import OpenAIEmbeddings

/var/folders/hq/q_jg97r105396k0c5t15jksc0000gq/T/ipykernel_10888/3473827221.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## STEP 0 - Load PDF into text Format

In [2]:
text_data = PyPDFLoader("NovaS.pdf").load()

for doc in text_data:
    doc.metadata["author"] = "Adarsh Maskar"
    
text_data

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Adarsh Maskar', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow b

## Step-1 - Creating Chunks of the text data

In [3]:
splitter  = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

chunks = splitter.split_documents(text_data)
len(chunks)

101

## Step-2 and 3 - Creating emmedings and Storing in vector DB 

In [4]:
#embed_model = OpenAIEmbeddings( model="text-embedding-3-small")
#embedded_chunks = embed_model.embed_documents(i.page_content for i in chunks)

from langchain_community.vectorstores import Chroma

In [5]:
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [6]:
chroma_db = Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db")

## Step-4 : Connection and Retrival

In [7]:
chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

/var/folders/hq/q_jg97r105396k0c5t15jksc0000gq/T/ipykernel_10888/3816121341.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [9]:
chroma_db_con.similarity_search("By 2019, how many employees were there?", k=3)

[Document(metadata={'page_label': '1', 'author': 'Adarsh Maskar', 'producer': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'moddate': '2026-03-31T11:24:15-03:00', 'total_pages': 3, 'source': 'NovaS.pdf', 'page': 0, 'creator': 'Microsoft® Word for Microsoft 365'}, page_content='By the beginning of 2019, the organization had grown to more than fifteen employees. This'),
 Document(metadata={'author': 'Adarsh Maskar', 'page_label': '2', 'creationdate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'moddate': '2026-03-31T11:24:15-03:00', 'page': 1, 'creator': 'Microsoft® Word for Microsoft 365', 'producer': 'Microsoft® Word for Microsoft 365'}, page_content='During 2023, the organization experienced steady growth. The number of employees'),
 Document(metadata={'author': 'Adarsh Maskar', 'producer': 'Microsoft® Word for Microsoft 365', 'page': 1, 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'page_label': '2', 'cre

## Step- 5 :-  LLM Integration and Answer Generation

In [ ]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)
#llm.invoke("What is name of the organization?")

AIMessage(content='I don’t have enough information — which organization are you asking about? Could you paste the text, image, screenshot, website link, or describe where you saw it?\n\nIf you want to find an organization name yourself, try these quick steps:\n- Check the letterhead, email domain, footer of the webpage, or logo for a full name.\n- Look for a registration number (EIN, company number) and search national registries (e.g., Companies House UK, Secretary of State business search in the relevant US state).\n- Use OpenCorporates, Guidestar/IRS Exempt Organizations (US charities), or the Charity Commission (UK) for nonprofits.\n- Do a reverse image search on the logo (Google Images) or search the domain with WHOIS.\n- Search LinkedIn, Google, or BBB for the name displayed in profiles or listings.\n\nTell me what you have (text, link, image, country) and I’ll help identify the organization.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_to

In [22]:
user_query = input("Enter your query: ")
relevant_chunks = chroma_db_con.similarity_search(user_query, k=2)

rel_chunks_content = []

for i, chunk in enumerate(relevant_chunks):
    rel_chunks_content.append(chunk.page_content)

llm.invoke(f"{user_query} Use the following context to answer the question: {rel_chunks_content}")

AIMessage(content='By the beginning of 2019 the organization had grown to more than 15 employees.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 64, 'total_tokens': 219, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E2F01GKYv48kpnIRK4EKIOzPazM5V', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f6ac9-8950-7ec2-b0fa-93d15d179226-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 155, 'total_tokens': 219, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}})